# Financial Transaction Fraud Detection
**Project 3 — Full Data Analytics Pipeline**
**Author:** Matile Durin Mohwaduba

---

## Business Problem

Financial fraud is one of the most damaging and costly threats facing banks, payment processors, and their customers. Every fraudulent transaction represents a direct financial loss — and more importantly, a breach of the trust customers place in financial institutions to keep their money safe.

This project analyses a real-world financial transactions dataset to answer two connected business questions:

> **1. What patterns exist in suspicious financial transactions — and can we identify them from transaction behaviour alone?**
> **2. Can we build a machine learning model that automatically flags suspicious transactions before they cause harm?**

The ability to detect fraud automatically has enormous financial value:
- **For financial institutions:** Automated detection catches threats at scale that human review cannot
- **For customers:** Suspicious transactions can be blocked or flagged in real time before money is lost
- **For regulators:** A documented, auditable flagging system demonstrates compliance with financial security standards

### Dataset Overview
The dataset is sourced from Kaggle's Financial Transactions dataset (`saidaminsaidaxmadov/financial-transactions`) and contains real-world style transaction records across multiple accounts, channels, and product types.

| Column | Description |
|--------|-------------|
| `TransactionID` | Unique identifier for each transaction |
| `AccountID` | The account that initiated the transaction |
| `TransactionDate` | Date the transaction occurred |
| `TransactionType` | Credit or Debit |
| `TransactionChannel` | How the transaction was made (Web, ATM, Mobile) |
| `ProductID` | The financial product associated with the transaction |
| `Status` | Whether the transaction succeeded or failed |
| `TransactionAmount` | The raw amount — including negative values for suspicious transactions |

### Analytical Pipeline

| Step | Task | Status |
|------|------|--------|
| 1 | Data Collection & Dataset Understanding | Complete |
| 2 | Data Cleaning & Preprocessing | Complete |
| 3 | Exploratory Data Analysis (EDA) | Complete |
| 4 | Data Visualisation | Complete |
| 5 | Predictive Modelling — Fraud Detection AI | Complete |

---

## Step 1: Data Collection & Dataset Understanding

### Goal
Load the dataset from Kaggle and understand its full structure — what files exist, what columns are available, how many records we are working with, and what each feature represents.

### Why Kaggle?
Kaggle is one of the world's largest repositories of real-world datasets used by data scientists and analysts globally. Loading directly from Kaggle via `kagglehub` ensures we are working with the original, unmodified source data — which is essential for reproducibility and credibility.

### Libraries used
- `kagglehub` — Kaggle's official Python library for downloading datasets programmatically
- `pandas` — Python's primary data manipulation library
- `os` — Python's built-in library for navigating file system paths

In [ ]:
print("Hello World - Matile Durin's Project 3 Initiated")

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("saidaminsaidaxmadov/financial-transactions")

print("Path to dataset files:", path)

**Observation:** The dataset downloads successfully from Kaggle and the local file path is returned. In the next cell we inspect what files are inside the downloaded folder — Kaggle datasets often contain multiple CSV files, and we need to identify the correct one before loading.

In [ ]:
import pandas as pd
import os

# 1. Look inside the Kaggle folder you just downloaded
files_in_folder = os.listdir(path)
print("Files found in the vault:", files_in_folder)

# 2. Find the exact CSV file and get its full location
csv_filename = [f for f in files_in_folder if f.endswith('.csv')][0]
full_file_path = os.path.join(path, csv_filename)

# 3. Load the financial data into the pandas engine
df = pd.read_csv(full_file_path)

print("\n--- Financial Data Loaded Successfully! ---")
print("Dataset Shape (Rows, Columns):", df.shape)
print("\n--- Detailed Column Breakdown ---")
df.info()

**Observation:** The folder contains multiple files. We target the transaction ledger specifically in the next cell, since that is the file containing the individual transaction records we need for fraud detection analysis.

In [ ]:
import pandas as pd
import os

# 1. Look inside the Kaggle folder you just downloaded
files_in_folder = os.listdir(path)

# 2. Target the specific transaction ledger
target_file = 'FactTransaction.csv'
full_file_path = os.path.join(path, target_file)

# 3. Load the actual financial transactions into the pandas engine
df_transactions = pd.read_csv(full_file_path)

print(f"--- {target_file} Loaded Successfully! ---")
print("Dataset Shape (Rows, Columns):", df_transactions.shape)
print("\n--- Detailed Column Breakdown ---")
df_transactions.info()

**Observations:**
- `FactTransaction.csv` is the core transaction ledger — the naming convention ("Fact" table) is borrowed from data warehouse design, where a Fact table stores measurable events (transactions) as opposed to Dimension tables which store descriptive attributes
- The dataset contains transaction records across multiple columns covering identity (`TransactionID`, `AccountID`), timing (`TransactionDate`), behaviour (`TransactionType`, `TransactionChannel`), and financial value (`TransactionAmount`)
- Critically, `TransactionAmount` is numeric — which means we can immediately apply mathematical rules to detect anomalies
- We note that `TransactionAmount` may contain **negative values**, which in a legitimate ledger would represent reversals or refunds — but in this context will be treated as our primary fraud signal

---

## Step 2: Data Cleaning & Preprocessing

### Goal
Prepare the dataset for analysis by resolving data quality issues and engineering the features needed for fraud detection.

### Why this step is especially important for financial data
In financial analysis, data quality is not just a technical concern — it is a compliance and security concern. An analysis built on duplicate records, failed transactions, or structurally inconsistent data would produce fraud flags that are unreliable and potentially misleading. Clean financial data is the foundation of a trustworthy detection system.

This step has three phases:
1. **Security audit** — check for duplicates and understand the data's structure
2. **Anomaly investigation** — identify and understand negative transaction amounts
3. **Feature engineering** — create the `Suspicious_Flag` and `True_Amount` columns that power the model

### 2.1 Security Audit — Duplicates & Structure

In [ ]:
# 1. Check for duplicate transactions (Security audit)
duplicates = df_transactions.duplicated().sum()
print("--- Security Audit ---")
print(f"Duplicate Transactions Found: {duplicates}")

# 2. Uncover the structure of the money movement
print("\n--- Financial Channels & Status ---")
print("Types of Transactions:", df_transactions['TransactionType'].unique())
print("Channels Used:", df_transactions['TransactionChannel'].unique())
print("Transaction Statuses:", df_transactions['Status'].unique())

# 3. Understand the scale of the money
print("\n--- Transaction Value Scale ---")
print(f"Smallest Swipe: \${df_transactions['TransactionAmount'].min():.2f}")
print(f"Largest Swipe: \${df_transactions['TransactionAmount'].max():.2f}")
print(f"Average Swipe: \${df_transactions['TransactionAmount'].mean():.2f}")

**Observations — key findings from the audit:**

- **Duplicate transactions:** Zero duplicates found — each transaction record is unique, as expected in a properly maintained financial ledger
- **Transaction types:** Two types only — Credit (money in) and Debit (money out) — a clean binary structure
- **Channels:** Three channels — Web, ATM, and Mobile — representing the full range of modern banking interaction points
- **Transaction statuses:** Multiple statuses exist in the raw data. This is important: we should only analyse transactions that actually completed. Failed or pending transactions represent a different data population and should be separated before analysis
- **Amount range:** The presence of negative values in `TransactionAmount` is the critical finding. In standard double-entry bookkeeping, negative amounts can represent legitimate reversals — but in this dataset, they are the fraud signal we will engineer around

### 2.2 Isolate Successful Transactions & Investigate the Anomaly

In [ ]:
# 1. Filter out the noise: We only want to analyze money that actually successfully moved
df_success = df_transactions[df_transactions['Status'] == 'Success'].copy()
print(f"Successful Transactions Isolated: {len(df_success)} records")

# 2. Investigate the anomaly: Hunt down the negative money
print("\n--- Investigating the Negative Money ---")
negative_swipes = df_success[df_success['TransactionAmount'] < 0]
print(f"Total Negative Transactions Found: {len(negative_swipes)}")

# 3. Check the accounting rules: Are all Debits negative and Credits positive?
print("\n--- Money Flow by Transaction Type ---")
# We will look at the minimum and maximum values for both Credits and Debits
money_flow = df_success.groupby('TransactionType')['TransactionAmount'].agg(['min', 'max', 'mean']).round(2)
print(money_flow)

**Observations — the anomaly confirmed:**

- **Successful transactions isolated:** We retain only completed transactions for analysis. This is the correct population for fraud detection — we cannot meaningfully flag a transaction that never completed
- **Negative transactions found:** A significant number of successful transactions carry negative amounts. In a clean accounting system, a successful transaction should always represent a positive monetary movement in one direction — either money in (Credit) or money out (Debit)
- **The accounting rule violation:** The `groupby` breakdown reveals whether negative amounts appear in both Credit and Debit types, or are concentrated in one. A negative Credit (money supposedly coming in, but recorded as negative) has no legitimate accounting explanation — it is structurally suspicious
- **Conclusion:** Negative `TransactionAmount` values on successful transactions represent our fraud signal. They do not conform to standard financial accounting rules and will be flagged accordingly

### 2.3 Feature Engineering — Building the Fraud Detection Columns

This is the most analytically important step in the cleaning phase. We create two new columns that transform raw financial data into a structure the model can learn from:

- **`Suspicious_Flag`** — a binary label: 1 if the transaction amount is negative (suspicious), 0 if positive (normal). This becomes our target variable for the classifier
- **`True_Amount`** — the absolute value of `TransactionAmount`. This standardises all amounts to a positive scale so the model can compare transaction sizes fairly, regardless of sign
- We then **drop `TransactionAmount`** — the original column has done its job. Keeping it alongside `True_Amount` would create redundant, confusing information for the model

In [ ]:
# 1. Feature Engineering: Create a 'Suspicious' security flag (1 for Yes, 0 for No)
df_success['Suspicious_Flag'] = (df_success['TransactionAmount'] < 0).astype(int)

# 2. Standardize the money: Convert all negative amounts to their true absolute value
df_success['True_Amount'] = df_success['TransactionAmount'].abs()

# 3. Drop the old confusing column so it doesn't distract the AI
df_clean_money = df_success.drop('TransactionAmount', axis=1)

# 4. Verify our new security perimeter
print("--- Security Feature Engineering Complete ---")
print(f"Suspicious Transactions Flagged for AI: {df_clean_money['Suspicious_Flag'].sum()}")
print("\n--- Cleaned True Money Flow ---")
print(f"Smallest True Amount: \${df_clean_money['True_Amount'].min():.2f}")
print(f"Largest True Amount: \${df_clean_money['True_Amount'].max():.2f}")
print(f"Average True Amount: \${df_clean_money['True_Amount'].mean():.2f}")

**Observations — feature engineering results:**

- **3,760 transactions flagged as suspicious** out of 9,533 successful transactions — approximately **39.4% of all completed transactions carry a suspicious flag**. This is a striking finding in itself: nearly 4 in every 10 successful transactions in this dataset are structurally anomalous
- **True_Amount range:** $11.07 to $10,000.00, with an average of $2,510.41 — a wide spread that suggests the suspicious transactions are not concentrated at one value point but distributed across all transaction sizes
- **Cleaning summary:**

| Issue | Finding | Action Taken |
|-------|---------|--------------|
| Duplicate records | 0 duplicates found | No action required |
| Failed transactions | Removed non-Success records | Filtered to Success only |
| Negative amounts | 3,760 suspicious transactions identified | Flagged as Suspicious_Flag = 1 |
| TransactionAmount | Raw column with mixed positive/negative | Replaced with True_Amount (absolute value) |

The dataset is now clean, structured, and ready for exploratory analysis.

---

## Step 3: Exploratory Data Analysis (EDA)

### Goal
Analyse the cleaned dataset to understand the distribution and behaviour of suspicious vs. safe transactions — before handing the data to a machine learning model.

### Why EDA matters before modelling
A fraud detection model trained without EDA is a black box built on assumptions. EDA ensures we:
- Understand the scale of the problem (how many flagged transactions, what proportion)
- Identify whether suspicious transactions follow detectable patterns (amount ranges, channels, transaction types)
- Confirm the data is structured correctly before the model is trained on it

The central question EDA must answer here: **Is there a meaningful, learnable pattern that separates suspicious from safe transactions — or are they indistinguishable?** If they are indistinguishable, no model will perform well. If clear patterns exist, the model has signal to learn from.

### 3.1 Suspicious Transaction Profile

We examine the suspicious transactions in detail — how many, what proportion, and how their amounts compare to normal transactions.

In [ ]:
print("--- Suspicious Transaction Summary ---")
print(f"Total Successful Transactions: {len(df_clean_money)}")
print(f"Safe Transactions (Flag = 0): {(df_clean_money['Suspicious_Flag'] == 0).sum()}")
print(f"Suspicious Transactions (Flag = 1): {(df_clean_money['Suspicious_Flag'] == 1).sum()}")
print(f"Suspicious Rate: {df_clean_money['Suspicious_Flag'].mean() * 100:.1f}%")

print("\n--- Amount Comparison: Safe vs Suspicious ---")
amount_profile = df_clean_money.groupby('Suspicious_Flag')['True_Amount'].agg(['min', 'max', 'mean']).round(2)
amount_profile.index = ['Safe (0)', 'Suspicious (1)']
print(amount_profile)

print("\n--- Channel Breakdown ---")
channel_breakdown = df_clean_money.groupby(['TransactionChannel', 'Suspicious_Flag']).size().unstack(fill_value=0)
channel_breakdown.columns = ['Safe', 'Suspicious']
print(channel_breakdown)

**Observations — EDA findings:**

- **39.4% suspicious rate** — this is far higher than typical real-world fraud rates (usually under 1%), which tells us this dataset is specifically constructed to study anomalous transaction behaviour rather than model a live banking environment. The patterns are real and learnable, but the prevalence is artificially elevated for analytical purposes
- **Amount profile:** Both safe and suspicious transactions span the full amount range ($11 to $10,000). The suspicious transactions are not simply "high-value outliers" — they appear at all transaction sizes. This means the model cannot rely on amount alone to detect them; it must learn from the combination of amount, channel, and transaction type
- **Channel distribution:** Examining how suspicious transactions distribute across Web, ATM, and Mobile channels reveals whether any channel is disproportionately exploited — a key insight for where security controls should be tightened

---

## Step 4: Data Visualisation

### Goal
Translate the numerical EDA findings into clear visual representations that communicate the threat landscape at a glance.

### Chart design decisions
Two charts are produced side by side, each revealing a different dimension of the fraud pattern:

- **Chart 1 (Count plot):** Shows the raw volume of safe vs. suspicious transactions — immediately communicating the scale of the problem
- **Chart 2 (Scatter plot):** Maps every individual transaction by sequence and amount, coloured by suspicion status — this reveals whether suspicious transactions cluster at particular points in time or at particular value ranges, or whether they are distributed randomly throughout the data

**Colour logic:** Green (0) for safe transactions, Red (1) for suspicious. The semantic meaning of the colours carries the message before the reader processes the labels — an intentional design decision to make the charts readable at a glance.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set a professional, clean theme
sns.set_theme(style="whitegrid", context="notebook")

# Red for Suspicious (Threats), Green for Safe (Normal Business)
security_colors = {0: '#2ecc71', 1: '#e74c3c'}

# Create a wide canvas for the dashboard
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: The Security Audit (Volume of Threats)
sns.countplot(data=df_clean_money, x='Suspicious_Flag', hue='Suspicious_Flag', ax=axes[0], palette=security_colors, legend=False)
axes[0].set_title('Transaction Volume: Safe vs Suspicious', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('Security Status', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Transactions', fontsize=12, fontweight='bold')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['0: Safe', '1: Suspicious'])

# Chart 2: The True Scatter Plot (Timeline of the attacks)
# We use the dataset's index (the sequence of transactions) to spread the data horizontally
sns.scatterplot(data=df_clean_money, x=df_clean_money.index, y='True_Amount', hue='Suspicious_Flag', ax=axes[1], palette=security_colors, alpha=0.7, s=50)
axes[1].set_title('The Threat Landscape: Timeline of Attacks', fontsize=14, fontweight='bold', pad=15)
axes[1].set_xlabel('Transaction Sequence (Timeline)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('True Transaction Amount ($)', fontsize=12, fontweight='bold')
axes[1].legend(title='Status', labels=['Safe', 'Suspicious'], bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

**Observations — what the charts reveal:**

**Count plot:** The volume difference between safe (5,773) and suspicious (3,760) transactions is clearly visible. Unlike a typical fraud dataset where suspicious transactions would be a tiny sliver, the bar heights here are in the same order of magnitude — confirming that this dataset is structured for fraud pattern analysis rather than live production modelling.

**Scatter plot — the more revealing chart:** The distribution of red (suspicious) and green (safe) points across the transaction timeline tells us whether the fraud is:
- **Clustered** — appearing in bursts at specific time periods (suggesting organised, coordinated attacks)
- **Random** — scattered evenly throughout the timeline (suggesting individual, opportunistic anomalies)
- **Amount-correlated** — concentrated at high or low transaction values

The pattern revealed here directly informs how the model should be interpreted and where a real-world security team should focus their investigation.

**Business implication:** If suspicious transactions are distributed across all channels, amounts, and time periods without obvious clustering, this is actually the hardest fraud scenario to detect — and the strongest argument for an automated ML-based detection system over manual review.

---

## Step 5: Predictive Modelling — Fraud Detection AI

### Goal
Build a machine learning classifier that can automatically identify suspicious transactions based on their behavioural patterns — without being told in advance which transactions are fraudulent.

### Model selection: Random Forest Classifier
We use a **Random Forest Classifier** for the following reasons:
- It handles a mix of numeric (`True_Amount`) and categorical (`TransactionType`, `TransactionChannel`) features effectively after encoding
- It is robust to imbalanced classes and does not require the data to follow a normal distribution
- It captures complex, non-linear decision boundaries — important when fraud patterns are not simply "high amount = suspicious"
- It provides an interpretable classification report showing precision and recall separately, which matters more than raw accuracy in fraud detection

### Why accuracy alone is not enough for fraud detection
In fraud detection, **precision** and **recall** are more important than overall accuracy:
- **Precision:** Of all transactions the model flags as suspicious, how many actually are? Low precision = too many false alarms, wasting investigator time
- **Recall:** Of all truly suspicious transactions, how many does the model catch? Low recall = missed fraud slipping through undetected

A model that flags everything as suspicious has 100% recall but 0% usefulness. We need both metrics to be high.

### Pipeline
1. **Drop non-predictive columns** — `TransactionID`, `AccountID`, and `TransactionDate` are identifiers and timestamps. They do not contain behavioural patterns the model should learn from
2. **Encode categoricals** — convert `TransactionType` and `TransactionChannel` text values into numeric 1s and 0s using `get_dummies`
3. **Separate X and y** — features (behavioural clues) from target (`Suspicious_Flag`)
4. **80/20 train-test split** — the model trains on 80% and is evaluated on the hidden 20% it has never seen
5. **Train and evaluate** — fit the classifier, make predictions, and read the full performance report

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Clean the Noise: The AI doesn't need to know the specific 'TransactionID' or 'Date'
# to spot a behavioral pattern, so we drop them to keep it focused.
features_to_drop = ['TransactionID', 'AccountID', 'TransactionDate']
df_ai = df_clean_money.drop(columns=features_to_drop)

# 2. Translate Text to Math: The AI only reads numbers, so we convert text like 'Web' or 'ATM' into 1s and 0s
df_encoded = pd.get_dummies(df_ai, drop_first=True)

# 3. Separate the Target (The Security Flag) from the Clues (Amount, Channel, Type)
y = df_encoded['Suspicious_Flag']
X = df_encoded.drop('Suspicious_Flag', axis=1)

# 4. Split the Vault: Give the AI 80% of the data to study, keep 20% hidden to test it
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Hire the Security Guard: Initialize and train the Classifier AI
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 6. Run the Drill: Ask the AI to evaluate the hidden 20% of the data
predictions = model.predict(X_test)

# 7. Read the Performance Review: How many threats did it successfully catch?
accuracy = accuracy_score(y_test, predictions)

print("--- Digital Security Guard Deployed ---")
print(f"Threat Detection Accuracy: {accuracy * 100:.2f}%\n")
print("--- Detailed Security Report ---")
# This report breaks down exactly how well it caught the 0s (Safe) and 1s (Suspicious)
print(classification_report(y_test, predictions))

### Model Results & Interpretation

**Overall accuracy:** The model achieves high accuracy on the test set. However, as noted above, we read the classification report carefully rather than relying on this headline number alone.

**Classification report breakdown:**

| Metric | Class 0 (Safe) | Class 1 (Suspicious) |
|--------|---------------|----------------------|
| Precision | How many flagged-safe were truly safe | How many flagged-suspicious were truly suspicious |
| Recall | How many safe transactions were correctly identified | How many suspicious transactions were caught |
| F1-Score | Harmonic mean of precision and recall | Harmonic mean of precision and recall |

**What strong performance looks like here:** Both classes should have high precision and recall. In fraud detection, a low recall on Class 1 (suspicious) is the critical failure mode — it means real suspicious transactions are escaping detection.

**Why the model performs well on this dataset:** The `Suspicious_Flag` was engineered directly from `TransactionAmount` (negative values = suspicious). The `True_Amount` column — the absolute value of the same field — retains a strong mathematical signal about which transactions were originally negative. The model effectively learns this relationship. In a production system, the fraud signal would be far more subtle and the model would need richer features to maintain this performance level.

**Important caveat:** This model should be treated as a proof-of-concept rather than a production-ready fraud detector. Real-world fraud detection requires:
- Continuous retraining as fraud patterns evolve
- Additional features (device fingerprinting, geolocation, behavioural history)
- Human review of model flags before action is taken

### Note on Duplicate Model Cell

The modelling pipeline appears twice in the original notebook. The second cell is identical to the first and produces the same output — it can be safely removed in a final submission. Only one training and evaluation run is needed.

---

## Step 4b: Security Dashboard (Visualisation — Detailed View)

The charts below provide a visual security dashboard — complementing the model's numerical output with an intuitive picture of the threat landscape across the full dataset.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Clean the Noise: Drop IDs and Dates to keep the AI focused on patterns
features_to_drop = ['TransactionID', 'AccountID', 'TransactionDate']
df_ai = df_clean_money.drop(columns=features_to_drop)

# 2. Translate Text to Math for the machine
df_encoded = pd.get_dummies(df_ai, drop_first=True)

# 3. Separate the Target (Security Flag) from the Clues
y = df_encoded['Suspicious_Flag']
X = df_encoded.drop('Suspicious_Flag', axis=1)

# 4. Split the Vault: 80% to study, 20% to test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Train the AI Security Guard
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 6. Evaluate Performance
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print("--- Digital Security AI Deployed ---")
print(f"Threat Detection Accuracy: {accuracy * 100:.2f}%\n")
print("--- Detailed Security Report ---")
print(classification_report(y_test, predictions))

---

## Final Step: Export Cleaned Dataset

The fully cleaned and feature-engineered dataset is saved as a CSV for future use, sharing, or further analysis. This is the file that contains the `Suspicious_Flag` and `True_Amount` columns — the two engineered features that made fraud detection possible.

Setting `index=False` ensures the pandas row index does not appear as an unwanted extra column in the exported file.

In [ ]:
from google.colab import files

# Save the pristine, security-flagged dataset to the subconscious memory
df_clean_money.to_csv('Cleaned_Financial_Transactions.csv', index=False)

# Instantly hand it over to your phone's downloads
files.download('Cleaned_Financial_Transactions.csv')

---

## Project Summary

| Finding | Detail |
|---------|--------|
| Dataset source | Kaggle — Financial Transactions (`FactTransaction.csv`) |
| Raw records loaded | 10,000+ transaction records |
| Successful transactions analysed | 9,533 |
| Suspicious transactions flagged | 3,760 (39.4% of successful transactions) |
| Fraud signal engineered from | Negative `TransactionAmount` values |
| Model type | Random Forest Classifier |
| Features used | True_Amount, TransactionType, TransactionChannel, ProductID |
| Evaluation metrics | Accuracy, Precision, Recall, F1-Score |

**Conclusion:** This project demonstrates a complete fraud detection analytics pipeline — from raw Kaggle data through feature engineering, exploratory analysis, visualisation, and a working machine learning classifier. The central technical insight is that negative transaction amounts on successful transactions represent a clear, engineered fraud signal — and a Random Forest Classifier can learn to detect this pattern with high accuracy. The broader business lesson is that automated fraud detection at scale is not just possible with machine learning — it is necessary, given the volume of transactions that financial institutions process daily.